In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/FIAP/Machine Learning & Data Science/2026/Datasets"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
clientes = pd.read_csv(path + "/sv_clientes.csv", parse_dates=["data_adesao", "data_cancelamento"])
pagamentos = pd.read_csv(path + "/sv_pagamentos.csv", parse_dates=["data_pagamento"])
consumo = pd.read_csv(path + "/sv_consumo.csv", parse_dates=["data_consumo"])
marketing_atr = pd.read_csv(path + "/sv_marketing_atribuicao.csv")
marketing_inv = pd.read_csv(path + "/sv_marketing_investimento.csv")
devices = pd.read_csv(path + "/sv_devices.csv")
consumo_detalhado = pd.read_csv(path + "/sv_consumo_detalhado.csv", parse_dates=["data_consumo"])

data_fim = pd.Timestamp("2025-01-01")

In [ ]:
clientes.sample(5)

,cliente_id,data_adesao,data_cancelamento,plano,idade,renda_mensal
840,841,2023-12-08,NaT,Basic,34.0,6755.56
405,406,2022-10-30,NaT,Basic,47.0,1620.87
1362,1363,2023-10-08,NaT,Basic,49.0,3919.09
162,163,2023-02-06,NaT,Standard,22.0,7328.08
728,729,2023-05-23,NaT,Standard,26.0,1114.64


In [ ]:
clientes['churn'] = clientes['data_cancelamento'].apply(lambda x: 1 if pd.notnull(x) else 0)

In [ ]:
pagamentos.sample(5)

,cliente_id,data_pagamento,valor_pago,metodo_pagamento
15941,847,2023-10-01,49.9,credito
27723,1466,2024-06-01,29.9,debito
24113,1276,2024-05-01,29.9,credito
12970,691,2024-07-01,79.9,debito
12576,675,2022-10-01,79.9,pix


In [ ]:
consumo.sample(5)

,cliente_id,data_consumo,minutos_consumidos,categoria
44839,396,2023-04-24,83.4,Serie
68626,615,2022-02-01,154.6,Documentario
69795,624,2024-12-12,135.0,Infantil
123915,1100,2024-03-05,152.5,Infantil
144971,1288,2023-12-10,255.7,Infantil


In [ ]:
consumo_detalhado.sample(5)

,cliente_id,data_consumo,categoria,titulo,minutos_consumidos,completion_rate,rating_implicito
10778,111,2024-07-10,Filme,The Dark Knight,92.6,0.74,4.40
11432,116,2025-04-01,Serie,Stranger Things,2.5,0.02,2.51
52421,538,2025-09-02,Documentario,Making a Murderer,173.4,1.00,4.79
53789,551,2023-12-21,Filme,The Matrix,7.7,0.08,2.87
82847,838,2025-02-03,Serie,Stranger Things,0.0,0.00,3.02


In [ ]:
marketing_atr.sample(5)

,cliente_id,data_atribuicao,canal
789,790,2024-03-04,Organico
205,206,2022-09-08,Meta
1214,1215,2022-01-29,Meta
521,522,2023-08-29,Email
37,38,2024-05-10,Meta


In [ ]:
marketing_inv.sample(5)

,mes,canal,investimento
116,2024-06-01,Google,56380.77
2,2022-01-01,TikTok,55624.56
108,2024-04-01,Google,48910.09
100,2024-02-01,Google,43711.28
71,2023-06-01,Email,52378.93


In [ ]:
devices.sample(5)

,cliente_id,tipo_device,data_ativacao_device
1102,545,Tablet,2024-03-15
1857,923,SmartTV,2024-02-16
2879,1447,Desktop,2022-07-02
1896,946,Tablet,2024-03-16
1293,642,Tablet,2023-06-21


In [ ]:
import pandas as pd
import numpy as np

def auditoria_temporal(clientes, consumo_det):


    DATA_FIM = pd.Timestamp("2025-01-01")
    clientes["data_fim"] = clientes["data_cancelamento"].fillna(DATA_FIM)

    inconsistencias = []

    # -----------------------------------
    # 1. Cancelamento antes da adesão
    # -----------------------------------
    invalido_cancel = clientes[
        (clientes["data_cancelamento"].notna()) &
        (clientes["data_cancelamento"] < clientes["data_adesao"])
    ]

    inconsistencias.append({
        "tipo": "cancelamento_antes_adesao",
        "quantidade": len(invalido_cancel)
    })

    # -----------------------------------
    # 2. Consumo fora da vigência
    # -----------------------------------
    consumo_merge = consumo_det.merge(
        clientes[["cliente_id","data_adesao","data_fim"]],
        on="cliente_id",
        how="left"
    )

    consumo_fora = consumo_merge[
        (consumo_merge["data_consumo"] < consumo_merge["data_adesao"])
    ]

    inconsistencias.append({
        "tipo": "consumo_fora_vigencia",
        "quantidade": len(consumo_fora)
    })

    # -----------------------------------
    # 3. Gap extremo de inatividade (>180 dias)
    # -----------------------------------
    consumo_sorted = consumo_det.sort_values(["cliente_id","data_consumo"])
    consumo_sorted["diff_dias"] = consumo_sorted.groupby("cliente_id")["data_consumo"].diff().dt.days

    gaps_extremos = consumo_sorted[consumo_sorted["diff_dias"] > 180]

    inconsistencias.append({
        "tipo": "gap_extremo_inatividade",
        "quantidade": len(gaps_extremos)
    })

    # -----------------------------------
    # 4. Clientes sem consumo
    # -----------------------------------
    clientes_sem_consumo = set(clientes["cliente_id"]) - set(consumo_det["cliente_id"])

    inconsistencias.append({
        "tipo": "clientes_sem_consumo",
        "quantidade": len(clientes_sem_consumo)
    })

    # -----------------------------------
    # Resultado
    # -----------------------------------
    df_resultado = pd.DataFrame(inconsistencias)

    print("\n===== RELATÓRIO DE QUALIDADE TEMPORAL =====")
    print(df_resultado)
    print("===========================================\n")

    return df_resultado

In [ ]:
auditoria_temporal(clientes, consumo_detalhado)


===== RELATÓRIO DE QUALIDADE TEMPORAL =====
                        tipo  quantidade
0  cancelamento_antes_adesao           0
1      consumo_fora_vigencia           0
2    gap_extremo_inatividade           0
3       clientes_sem_consumo         300



,tipo,quantidade
0,cancelamento_antes_adesao,0
1,consumo_fora_vigencia,0
2,gap_extremo_inatividade,0
3,clientes_sem_consumo,300


## Feature Engineering

In [ ]:
# Clientes
clientes['data_fim'] = clientes['data_cancelamento'].fillna(data_fim)
clientes['tenure_dias'] = (clientes["data_fim"] - clientes["data_adesao"]).dt.days

In [ ]:
# Pagamentos
pag_agg = pagamentos.groupby("cliente_id").agg(
    total_pago=("valor_pago", "sum"),
    ticket_medio=("valor_pago", "mean"),
    num_pagamentos=("valor_pago", "count")
).reset_index()

In [ ]:
# Consumo detalhado
data_fim_consumo = consumo_detalhado["data_consumo"].max()

cons_global = consumo_detalhado.groupby("cliente_id").agg(
    total_minutos = ("minutos_consumidos", "sum"),
    media_minutos = ("minutos_consumidos", "mean"),
    max_minutos = ("minutos_consumidos", "max"),
    min_minutos = ("minutos_consumidos", "min"),
    std_minutos = ("minutos_consumidos", "std"),
    total_conteudos = ("titulo", "count"),
    titulos_unicos = ("titulo", "nunique"),
    categorias_unicas = ("categoria", "nunique"),
    dias_ativo = ("data_consumo", "nunique"),
    media_completion = ("completion_rate", "mean"),
    media_rating = ("rating_implicito", "mean"),
    std_rating = ("rating_implicito", "std"),
    max_rating = ("rating_implicito", "max"),
    min_rating = ("rating_implicito", "min"),
    consumo_recente = ("data_consumo", lambda x: (data_fim_consumo - x.max()).days)
).reset_index()

# Consumo por dia, por conteudo
cons_global['minutos_por_dia_ativo'] = cons_global["total_minutos"] / cons_global["dias_ativo"]
cons_global['minutos_por_conteudo'] = cons_global["total_minutos"] / cons_global["total_conteudos"]

# Engajamento
cons_global["taxa_rewatch"] = (
    consumo_detalhado.groupby("cliente_id")["titulo"].count() /
    consumo_detalhado.groupby("cliente_id")["titulo"].nunique()
).reindex(cons_global["cliente_id"]).values

cons_global["completition_alta_pct"] = (
    consumo_detalhado[consumo_detalhado['completion_rate']>=0.8]
    .groupby("cliente_id")["titulo"]
    .count()
    .reindex(cons_global["cliente_id"])
    .fillna(0)
    .div(cons_global["total_conteudos"].values)
    .values
)

cons_global["rating_alta_pct"] = (
    consumo_detalhado[consumo_detalhado['rating_implicito']>=4]
    .groupby("cliente_id")["titulo"]
    .count()
    .reindex(cons_global["cliente_id"])
    .fillna(0)
    .div(cons_global["total_conteudos"].values)
    .values
)

# Features temporais
consumo_detalhado['mes'] = consumo_detalhado['data_consumo'].dt.to_period('M')
consumo_mensal = consumo_detalhado.groupby(["cliente_id", "mes"]).agg(
    minutos_mes = ("minutos_consumidos", "sum")
).reset_index()

# tendencia de consumo (ultimo mes vs mes anterior)
ultima_data = consumo_mensal["mes"].max()
ultimo_mes = consumo_mensal[consumo_mensal["mes"] == ultima_data]
media_anterior = consumo_mensal[consumo_mensal["mes"] < ultima_data].groupby("cliente_id")["minutos_mes"].mean().reset_index()

tendencia = ultimo_mes.merge(media_anterior, on="cliente_id", how="left", suffixes=("_ultimo", "_media"))
tendencia["variacao_consumo"] = (
    (tendencia["minutos_mes_ultimo"] - tendencia["minutos_mes_media"]) /
    tendencia["minutos_mes_media"]
)

cons_global = cons_global.merge(
    tendencia[["cliente_id", "variacao_consumo"]],
    on="cliente_id",
    how="left"
)

# Features por categoria (filmes, series ...)
cat_agg = consumo_detalhado.groupby(["cliente_id", "categoria"]).agg(
    minutos_cat = ("minutos_consumidos", "sum"),
    rating_cat = ("rating_implicito", "mean"),
    qtd_cat = ("titulo", "count")
).reset_index()

# Categoria mais vista
idx_max_view = cat_agg.groupby("cliente_id")["minutos_cat"].idxmax()
cat_favorita_view = cat_agg.loc[idx_max_view, ["cliente_id", "categoria"]]
cat_favorita_view = cat_favorita_view.rename(columns={"categoria": "categoria_mais_vista"})

#categoria melhor avaliada
idx_max_rating = cat_agg.groupby("cliente_id")["rating_cat"].idxmax()
cat_favorita_rating = cat_agg.loc[idx_max_view, ["cliente_id", "categoria"]]
cat_favorita_rating = cat_favorita_rating.rename(columns={"categoria": "categoria_melhor_avaliada"})

cons_global = cons_global.merge(cat_favorita_view, on="cliente_id", how="left")
cons_global = cons_global.merge(cat_favorita_rating, on="cliente_id", how="left")

# One-Hot Encoding por categoria
cat_pivot = cat_agg.pivot(index="cliente_id", columns="categoria", values="minutos_cat").fillna(0)
cat_pivot = cat_pivot.div(cat_pivot.sum(axis=1), axis=0)
cat_pivot.columns = [f"pct_consumo_{c}" for c in cat_pivot.columns]

cons_global = cons_global.merge(cat_pivot.reset_index(), on="cliente_id", how="left")

# Diversificação categoria e profundidade
cons_global["diversificacao_categoria"] = cons_global["categorias_unicas"] / cons_global["total_conteudos"]
cons_global["engajamento_score"] = (
    cons_global["media_completion"] * 0.4 +
    cons_global["media_rating"] * 0.4 +
    (1 - cons_global["consumo_recente"] / 365)* 0.2
)

dev_agg = devices.groupby("cliente_id").agg(
    qtd_devices = ("tipo_device", "nunique"),
    device_major = ("tipo_device", lambda x: x.mode()[0] if not x.mode().empty else "Desconhecido")
).reset_index()

cons_agg = cons_global.copy()
cons_agg.shape
cons_agg.head(5)


,cliente_id,total_minutos,media_minutos,max_minutos,min_minutos,std_minutos,total_conteudos,titulos_unicos,categorias_unicas,dias_ativo,...,rating_alta_pct,variacao_consumo,categoria_mais_vista,categoria_melhor_avaliada,pct_consumo_Documentario,pct_consumo_Filme,pct_consumo_Infantil,pct_consumo_Serie,diversificacao_categoria,engajamento_score
0,1,14805.1,85.578613,407.7,0.2,80.250807,173,31,4,153,...,0.560694,-0.469490,Serie,Serie,0.148928,0.320707,0.070280,0.460085,0.023121,2.078516
1,2,15100.5,93.212963,640.5,1.0,92.606426,162,31,4,148,...,0.543210,0.400809,Filme,Filme,0.212894,0.377504,0.088593,0.321009,0.024691,2.090593
2,3,6174.2,78.154430,328.2,0.7,71.514825,79,25,4,76,...,0.544304,-0.131722,Serie,Serie,0.151161,0.274044,0.108840,0.465955,0.050633,2.071220
3,4,5958.1,93.095313,340.1,1.3,75.897904,64,26,4,57,...,0.625000,NaN,Serie,Serie,0.149914,0.211678,0.062788,0.575620,0.062500,1.824308
4,5,8633.2,81.445283,298.7,1.5,72.359582,106,26,4,97,...,0.575472,0.333350,Serie,Serie,0.140933,0.262255,0.172740,0.424072,0.037736,2.089962


In [ ]:
df = clientes.merge(pag_agg, on="cliente_id", how="left") \
             .merge(cons_agg, on="cliente_id", how="left") \
             .merge(dev_agg, on="cliente_id", how="left") \
             .merge(marketing_atr[["cliente_id", "canal"]], on="cliente_id", how="left")
df.fillna(0, inplace=True)

/tmp/ipython-input-1095/3860479140.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df.fillna(0, inplace=True)


In [ ]:
df.shape
df.sample(5)

,cliente_id,data_adesao,data_cancelamento,plano,idade,renda_mensal,data_fim,churn,tenure_dias,total_pago,...,categoria_melhor_avaliada,pct_consumo_Documentario,pct_consumo_Filme,pct_consumo_Infantil,pct_consumo_Serie,diversificacao_categoria,engajamento_score,qtd_devices,device_major,canal
720,721,2022-06-02,2023-11-21 00:00:00,Basic,17.0,6136.35,2023-11-21,1,537,508.30,...,Serie,0.140638,0.195263,0.101902,0.562197,0.047619,1.759229,1,Tablet,Email
1379,1380,2022-09-03,0,Basic,34.0,3592.22,2025-01-01,0,851,822.25,...,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,Tablet,Email
486,487,2023-11-20,0,Basic,34.0,1600.71,2025-01-01,0,408,418.60,...,Serie,0.196746,0.351579,0.068519,0.383156,0.036697,1.976233,1,Mobile,Google
1063,1064,2023-07-05,0,Premium,42.0,4673.53,2025-01-01,0,546,1438.20,...,Serie,0.096261,0.344214,0.085009,0.474516,0.032787,2.106436,1,Mobile,Organico
857,858,2023-01-23,0,Premium,60.0,4418.90,2025-01-01,0,709,1917.60,...,Serie,0.184861,0.245394,0.113575,0.456171,0.020619,2.052079,2,Desktop,TikTok


In [ ]:
df.to_csv(path + "/df.csv", index=False)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   cliente_id                 1500 non-null   int64         
 1   data_adesao                1500 non-null   datetime64[ns]
 2   data_cancelamento          1500 non-null   object        
 3   plano                      1500 non-null   object        
 4   idade                      1500 non-null   float64       
 5   renda_mensal               1500 non-null   float64       
 6   data_fim                   1500 non-null   datetime64[ns]
 7   churn                      1500 non-null   int64         
 8   tenure_dias                1500 non-null   int64         
 9   total_pago                 1500 non-null   float64       
 10  ticket_medio               1500 non-null   float64       
 11  num_pagamentos             1500 non-null   int64         
 12  total_